# Harmonic downgrading: aliasing stress test

A synthetic sky with power concentrated entirely **above** the output
bandlimit ($\ell > \ell_{\max}^{\mathrm{out}}$).  The correct low-resolution map
should be near-zero, so any residual power directly measures **aliasing leakage**.

## Method overview

`harmonic_ud_grade` implements **Planck 2015 XVI Eq. 1**:

$$
a^{\mathrm{out}}_{\ell m}
= \frac{p^{\mathrm{out}}_\ell}{p^{\mathrm{in}}_\ell}\,
  \frac{b^{\mathrm{out}}_\ell}{b^{\mathrm{in}}_\ell}\,
  a^{\mathrm{in}}_{\ell m}
$$

| # | Method | Description |
|---|--------|-------------|
| 1 | **ud_grade** | Pixel averaging (no harmonic analysis) |
| 2 | **Harmonic clip** | SHT truncation only (`pixwin=False, fwhm_out=0`) |
| 3 | **Harmonic + pixwin** | Pixel-window correction (`pixwin=True, fwhm_out=0`) |
| 4 | **Harmonic + pixwin + beam** | Full Eq. 1 (default `harmonic_ud_grade`) |
| 5 | **Smoothing + ud_grade** | Traditional: pre-smooth then pixel-average |

In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

COLORS = {
    "ud_grade":   "#d95f02",
    "clip":       "#1b9e77",
    "pixwin":     "#33a02c",
    "full":       "#1f78b4",
    "smooth_ud":  "#7570b3",
}
METHOD_LABELS = {
    "ud_grade":   "ud_grade",
    "clip":       "Harmonic clip",
    "pixwin":     "Harmonic + pixwin",
    "full":       "Harmonic + pixwin + beam",
    "smooth_ud":  "Smoothing + ud_grade",
}
METHOD_ORDER = ["ud_grade", "clip", "pixwin", "full", "smooth_ud"]
LINESTYLES = {
    "ud_grade":   "-",
    "clip":       "--",
    "pixwin":     "-.",
    "full":       "-",
    "smooth_ud":  ":",
}

## Synthetic input: high-$\ell$ bump

The input spectrum is a Gaussian bump centred well above
$\ell_{\max}^{\mathrm{out}} = 95$, with a small noise floor.
A correctly band-limited output should contain essentially no signal.

In [ ]:
nside_in  = 256
nside_out = 32
lmax_in   = 3 * nside_in - 1
lmax_out  = 3 * nside_out - 1

ell = np.arange(lmax_in + 1, dtype=float)
ell0 = lmax_out + 55          # bump centre at ell ~ 150
sigma_ell = 28
cl = 1e-4 * np.ones_like(ell)                        # noise floor
cl[2:] += np.exp(-0.5 * ((ell[2:] - ell0) / sigma_ell) ** 2)

np.random.seed(4321)
alm_in = hp.synalm(cl, lmax=lmax_in)
m_in   = hp.alm2map(alm_in, nside_in, verbose=False)

# Reference: bandlimit-truncated version (should be ~ noise floor only)
alm_ref = hp.almxfl(alm_in, np.ones(lmax_out + 1))
m_ref   = hp.alm2map(alm_ref, nside_out, verbose=False)

print(f"Reference map std = {np.std(m_ref):.4e} (should be very small)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(ell[2:], cl[2:], "k-", lw=1.2, label="Input $C_\\ell$")
ax.axvline(lmax_out, color="red", ls="--", lw=1.5,
           label=f"$\\ell_{{\\max}}^{{\\mathrm{{out}}}}={lmax_out}$")
ax.fill_betweenx([cl[2:].min() * 0.5, cl[2:].max() * 2],
                 0, lmax_out, color="green", alpha=0.08,
                 label="Output band (should be ~empty)")
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$C_\ell$")
ax.set_title("Input spectrum — nearly all power lies above the output bandlimit")
ax.legend(fontsize=9)
ax.set_xlim(0, lmax_in)
fig.tight_layout()
plt.show()

In [ ]:
# Smoothing FWHM for the traditional approach: match harmonic_ud_grade default
fwhm_smooth = 3 * hp.nside2resol(nside_out)

maps = {}
maps["ud_grade"]  = hp.ud_grade(m_in, nside_out)
maps["clip"]      = hp.harmonic_ud_grade(m_in, nside_out, pixwin=False, fwhm_out=0)
maps["pixwin"]    = hp.harmonic_ud_grade(m_in, nside_out, pixwin=True,  fwhm_out=0)
maps["full"]      = hp.harmonic_ud_grade(m_in, nside_out)
maps["smooth_ud"] = hp.ud_grade(hp.smoothing(m_in, fwhm=fwhm_smooth, verbose=False),
                                nside_out)

print(f"{'Method':<30s} {'std':>12s}")
print("-" * 44)
for key in METHOD_ORDER:
    print(f"{METHOD_LABELS[key]:<30s} {np.std(maps[key]):12.4e}")
print(f"{'Reference (bandlimited)':<30s} {np.std(m_ref):12.4e}")

## Downgraded maps

The **reference** map (top-left) is the bandlimit-truncated truth — it should
look near-zero because the input has almost no power below $\ell_{\max}^{\mathrm{out}}$.

- **ud_grade** folds high-$\ell$ power into all scales (aliasing is dramatic).
- All **harmonic methods** naturally suppress aliasing by operating only on
  $\ell \leq \ell_{\max}^{\mathrm{out}}$.
- **Smoothing + ud_grade** suppresses it too, because the Gaussian kills
  high-$\ell$ power before pixel averaging.

In [ ]:
all_maps = [m_ref] + [maps[k] for k in METHOD_ORDER]
all_titles = ["Reference (bandlimited)"] + [METHOD_LABELS[k] for k in METHOD_ORDER]

peak = max(np.nanmax(np.abs(m)) for m in all_maps)
vlim = float(np.round(peak, 4))

fig = plt.figure(figsize=(14, 8))
for i, (m, t) in enumerate(zip(all_maps, all_titles), 1):
    hp.projview(
        m,
        sub=(2, 3, i),
        title=t,
        min=-vlim,
        max=vlim,
        cmap="RdBu_r",
        graticule=True,
        xsize=400,
        cb_orientation="horizontal",
    )
fig.suptitle(
    f"Downgraded maps  (NSIDE {nside_in} → {nside_out}), "
    f"shared colour scale [−{vlim:.4f}, +{vlim:.4f}]",
    fontsize=13, y=1.01,
)
fig.tight_layout()
plt.show()

## Power spectra of downgraded maps

The reference spectrum (black solid) sits at the noise floor.
Any curve above it indicates aliased power that leaked through.

Note: the harmonic methods overlap almost perfectly at the reference
level — use the dashed/dotted line styles to distinguish them.

In [ ]:
cl_ref = hp.anafast(m_ref, lmax=lmax_out)
cl_methods = {k: hp.anafast(maps[k], lmax=lmax_out) for k in METHOD_ORDER}
ell_out = np.arange(lmax_out + 1, dtype=float)

fig, ax = plt.subplots(figsize=(8, 5))

ax.loglog(ell_out[2:], cl[2:lmax_out + 1], "k:", lw=0.8, alpha=0.5,
          label="Input $C_\\ell$ (in-band)")
ax.loglog(ell_out[2:], cl_ref[2:], "k-", lw=2, label="Reference (bandlimited)")

for key in METHOD_ORDER:
    ax.loglog(ell_out[2:], cl_methods[key][2:], lw=1.5,
              color=COLORS[key], ls=LINESTYLES[key],
              label=METHOD_LABELS[key])

ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$C_\ell$")
ax.set_title("Power spectra of downgraded maps")
ax.legend(fontsize=8, ncol=2, loc="upper right")
ax.set_xlim(2, lmax_out)
fig.tight_layout()
plt.show()

## Aliasing quantification

Two metrics quantify how much input power leaks into the output:

- **Aliasing fraction** = $\sigma(\mathrm{method}) / \sigma(\mathrm{input})$:
  what fraction of the input map's RMS appears in the output.
  For a perfect method this should equal $\sigma(\mathrm{ref}) / \sigma(\mathrm{input}) \approx 0$.
- **Spectral RMS** = $\sqrt{\langle C_\ell \rangle}$: the average power
  per multipole in the output band.

In [ ]:
std_in = np.std(m_in)
std_ref = np.std(m_ref)

rows = []
for key in METHOD_ORDER:
    s = np.std(maps[key])
    frac = s / std_in
    cl_m = cl_methods[key]
    spectral_rms = np.sqrt(np.mean(cl_m[2:]))
    rows.append((METHOD_LABELS[key], s, frac, spectral_rms))

spectral_rms_ref = np.sqrt(np.mean(cl_ref[2:]))

print(f"{'Method':<30s} {'std':>12s} {'alias frac':>12s} {'spectral RMS':>14s}")
print("-" * 70)
for label, s, frac, srms in rows:
    print(f"{label:<30s} {s:12.4e} {frac:12.4e} {srms:14.4e}")
print(f"{'Reference (bandlimited)':<30s} {std_ref:12.4e} "
      f"{std_ref/std_in:12.4e} {spectral_rms_ref:14.4e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

labels = [METHOD_LABELS[k] for k in METHOD_ORDER]
fracs  = [r[2] for r in rows]
colors = [COLORS[k] for k in METHOD_ORDER]

# Left: aliasing fraction (log scale to show small values)
bars = ax1.barh(labels, fracs, color=colors, edgecolor='0.3', linewidth=0.5)
ax1.axvline(std_ref / std_in, color="k", ls="--", lw=1, label="Reference level")
ax1.set_xscale('log')
ax1.set_xlabel(r"Aliasing fraction $\sigma_{\mathrm{method}} / \sigma_{\mathrm{input}}$")
ax1.set_title("RMS aliasing leakage (log scale)")
ax1.legend(fontsize=9)

# Right: spectral RMS
spec_vals = [r[3] for r in rows]
bars2 = ax2.barh(labels, spec_vals, color=colors, edgecolor='0.3', linewidth=0.5)
ax2.axvline(spectral_rms_ref, color="k", ls="--", lw=1, label="Reference level")
ax2.set_xscale('log')
ax2.set_xlabel(r"Spectral RMS $\sqrt{\langle C_\ell \rangle}$")
ax2.set_title("Average spectral power in output band (log scale)")
ax2.legend(fontsize=9)

fig.suptitle("Aliasing metrics (lower = better)", fontsize=13)
fig.tight_layout()
plt.show()

## Takeaways

| Observation | Implication |
|---|---|
| **ud_grade** leaks substantial aliased power into all scales | Never use for precision work |
| **All harmonic methods** suppress aliasing to the noise floor | SHT truncation is the key anti-aliasing mechanism |
| Pixwin and beam corrections make negligible difference here | They matter for spectral fidelity, not aliasing |
| **Smoothing + ud_grade** also suppresses aliasing well | The pre-smoothing kills high-$\ell$ power before pixel averaging |